# 手写transformer Decoder
面试过程中让写 transformers Decoder 一定要沟通清楚是写一个 **CausalLM decoder** 还是原版的，原版的比较复杂，一般也不会让写。这里的 Decoder 一般指的是 CausalLM，具体变化：
- 少了 encoder 部分的输入，所以也就没有了 encoder and decoder cross attention。
- 因为重点希望写 CausalLM，所以没有 Cross attention 和 也省略了 token embedding 这一步。

[详解transformer](https://zhuanlan.zhihu.com/p/454482273)

![](resource/transformer_achiv.png)

- 采用post-norm形式，请分析pre-norm与post-norm的区别
- norm采用layernorm，而不是batchnorm，请分析两者区别，以及为什么
- 分析MHA相对于self-attention的优势
- 分析postional encoding的原理和作用

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

# 单个decoder block
class SimpleDecoder(nn.Module):
    def __init__(self, hidden_dim, head_num, attetion_dropout_rate=0.1):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.head_dim = hidden_dim // head_num
        
        # layer(mha, ffn)
        # mha
        self.query_pro = nn.Linear(hidden_dim, hidden_dim)
        self.key_pro = nn.Linear(hidden_dim, hidden_dim)
        self.value_pro = nn.Linear(hidden_dim, hidden_dim)
        
        self.o_proj = nn.Linear(hidden_dim,hidden_dim)
        self.attention_dropout = nn.Dropout(attetion_dropout_rate)
        self.att_ln = nn.LayerNorm(hidden_dim, eps=1e-10)
        
        # ffn (升维度->降维度->ln)
        self.up_proj = nn.Linear(hidden_dim, hidden_dim * 4)
        self.down_proj = nn.Linear(hidden_dim*4, hidden_dim)
        self.act_fn = nn.GELU() # or ReLU()
        self.drop_ffn = nn.Dropout(attetion_dropout_rate)
        self.ffn_ln = nn.LayerNorm(hidden_dim, eps=1e-10)
        
    def attention_layer(self, X, Q,K,V, attention_mask=None):
        # K->(batch, head_num, head_dim, seq)
        attention_scores = Q @ K.transpose(-1,-2)
        attention_weight = attention_scores / math.sqrt(self.head_dim)
        
        # **自带的下三角矩阵以及attention_mask**
        if attention_mask is not None:
            attention_mask = attention_mask.tril()
            attention_weight = attention_weight.masked_fill(
                attention_mask == 0, float('-inf')
            )
        else:
            attention_mask = torch.ones_like(attention_mask).tril()
            attention_weight = attention_weight.masked_fill(
                attention_mask == 0, float('-inf')
            )
        attention_weight = F.softmax(attention_weight, dim=-1)
        attention_weight = self.attention_dropout(attention_weight)
        
        # mid_output :(b,head_num, seq, head_dim)
        mid_output = attention_weight @ V
        mid_output = mid_output.transpose(1,2).contiguous()
        batch, seq, _, _ = mid_output.size()
        # mid_output :(b,seq, head_num, head_dim)
        mid_output = mid_output.view(batch, seq, self.hidden_dim)
        output = self.o_proj(mid_output)
        return output
    def mha_block(self,X, attention_mask=None):
        # 定义MHA部分的计算
        # Q,K,V：(batch, seq, hidden_dim)
        Q = self.query_pro(X)
        K = self.key_pro(X)
        V = self.value_pro(X)
        
        # 对QKV进行分组，组数为head_num，每组维度为head_dim
        # QKV:(batch, head_num, seq, head_dim)
        batch, seq , _ = X.size()
        Q = Q.view(batch,seq, -1, self.head_dim).transpose(1,2)
        K = K.view(batch,seq, -1, self.head_dim).transpose(1,2)
        V = V.view(batch,seq, -1, self.head_dim).transpose(1,2)
        
        output = self.attention_layer(X, Q,K,V, attention_mask)
        # post-norm # (b,s,h)
        return self.att_ln(X + output)
    def ffn_block(self,X):
        # 定义FFN部分计算
        up = self.up_proj(X)
        down = self.down_proj(up)
        down = self.act_fn(down)
        down = self.drop_ffn(down)
        # post layernorm
        return self.ffn_ln(X + down)
        
        pass
    def forward(self, X, attention_mask=None):
        # X：(batch, seq, hidden_dim)
        mha_output = self.mha_block(X, attention_mask)
        ffn_output = self.ffn_block(X)
        return ffn_output

X = torch.randn(2,3,64)
net = SimpleDecoder(64,8)
mask = (
    torch.tensor([[1,1,1,1], [1,1,0,0], [1,1,1,0]])
    .unsqueeze(1)
    .unsqueeze(2)
    .repeat(1,8,4,1)
)
net(X, mask)


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.0.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "D:\miniconda\envs\py3.9\lib\runpy.py", line 197, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "D:\miniconda\envs\py3.9\lib\runpy.py", line 87, in _run_code
    exec(code, run_globals)
  File "D:\miniconda\envs\py3.9\lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "D:\miniconda\envs\py3.9\lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "D:\miniconda\envs\py3.9\lib\site-packages

RuntimeError: The size of tensor a (4) must match the size of tensor b (3) at non-singleton dimension 3